In [1]:
import csv
import os
import re
import requests

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

# Your Google Sheet (public share link)
SHEET_URL = "https://docs.google.com/spreadsheets/d/1lHUX4W8M2yKQSymul4Xu55jUri9fzgK2h7lHmQdPQoQ/export?format=csv"

# Output CSV filename
LOCAL_CSV = "complaints.csv"

# Folder to save downloaded images
IMAGE_DIR = "downloaded_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# ------------------------------------------------------------
# STEP 1 — DOWNLOAD GOOGLE SHEET AS CSV
# ------------------------------------------------------------

print("[+] Downloading CSV from Google Sheets...")
csv_data = requests.get(SHEET_URL)
csv_data.raise_for_status()

with open(LOCAL_CSV, "wb") as f:
    f.write(csv_data.content)

print(f"[+] Saved CSV → {LOCAL_CSV}")

# ------------------------------------------------------------
# STEP 2 — PARSE CSV AND DOWNLOAD IMAGES
# ------------------------------------------------------------

def extract_drive_id(url):
    """
    Extract Google Drive file ID from any of these formats:
    - https://drive.google.com/open?id=FILEID
    - https://drive.google.com/file/d/FILEID/view
    - https://drive.google.com/uc?id=FILEID
    """
    if "id=" in url:
        return url.split("id=")[1]
    m = re.search(r"/d/([^/]+)", url)
    if m:
        return m.group(1)
    return None

def download_drive_file(file_id, dest_path):
    """
    Download a Google Drive file using the 'uc' endpoint.
    Works for images, PDFs, etc.
    """
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = requests.get(download_url, stream=True)
    r.raise_for_status()

    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)

    return dest_path

print("[+] Reading CSV and downloading linked images...")

with open(LOCAL_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for idx, row in enumerate(reader, start=1):
        img_url = row.get("Upload Images or Documents", "").strip()

        if img_url:
            file_id = extract_drive_id(img_url)
            if file_id:
                out_path = os.path.join(IMAGE_DIR, f"row_{idx}_{file_id}.jpg")
                try:
                    download_drive_file(file_id, out_path)
                    print(f"    ✔ Downloaded image for row {idx} → {out_path}")
                except Exception as e:
                    print(f"    ✖ Failed to download image for row {idx}: {e}")
            else:
                print(f"    ✖ Could not extract Drive ID for row {idx}: {img_url}")
        else:
            print(f"    (Row {idx}) No image link found.")

print("[+] Done!")


[+] Downloading CSV from Google Sheets...
[+] Saved CSV → complaints.csv
[+] Reading CSV and downloading linked images...
    ✔ Downloaded image for row 1 → downloaded_images\row_1_1BMab59LFHNEhht-onFDYPEtf1P2n65DC.jpg
[+] Done!


In [2]:
import csv
import os
import re
import requests
from datetime import datetime

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

SHEET_URL = "https://docs.google.com/spreadsheets/d/1lHUX4W8M2yKQSymul4Xu55jUri9fzgK2h7lHmQdPQoQ/export?format=csv"
LOCAL_CSV = "complaints.csv"
IMAGE_DIR = "downloaded_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_filename(text):
    """
    Convert text into a filesystem-safe slug:
    - lowercase
    - alphanumeric + underscore only
    - collapse spaces
    """
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)       # remove special chars
    text = re.sub(r"\s+", "_", text)           # spaces → underscores
    return text


def extract_drive_id(url):
    if "id=" in url:
        return url.split("id=")[1]
    m = re.search(r"/d/([^/]+)", url)
    if m:
        return m.group(1)
    return None


def download_drive_file(file_id, dest_path):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)

    return dest_path


def next_available_filename(base_path):
    """
    If base_path exists, append _1, _2, etc.
    """
    if not os.path.exists(base_path):
        return base_path

    root, ext = os.path.splitext(base_path)
    counter = 1

    while True:
        candidate = f"{root}_{counter}{ext}"
        if not os.path.exists(candidate):
            return candidate
        counter += 1


# ------------------------------------------------------------
# STEP 1 — DOWNLOAD CSV
# ------------------------------------------------------------

print("[+] Downloading CSV...")
csv_data = requests.get(SHEET_URL)
csv_data.raise_for_status()

with open(LOCAL_CSV, "wb") as f:
    f.write(csv_data.content)

print(f"[+] Saved CSV → {LOCAL_CSV}")

# ------------------------------------------------------------
# STEP 2 — PROCESS ROWS + DOWNLOAD IMAGES
# ------------------------------------------------------------

print("[+] Processing rows...")

with open(LOCAL_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for idx, row in enumerate(reader, start=1):
        img_url = row.get("Upload Images or Documents", "").strip()
        nature = row.get("Nature of Complaint", "unknown").strip()
        date_incident = row.get("Date of Incident", "").strip()

        # Normalize date format
        try:
            parsed_date = datetime.strptime(date_incident, "%m/%d/%Y")
            date_slug = parsed_date.strftime("%Y-%m-%d")
        except:
            date_slug = "unknown_date"

        nature_slug = safe_filename(nature)

        if img_url:
            file_id = extract_drive_id(img_url)
            if file_id:
                base_filename = f"{nature_slug}_{date_slug}.jpg"
                base_path = os.path.join(IMAGE_DIR, base_filename)

                final_path = next_available_filename(base_path)

                try:
                    download_drive_file(file_id, final_path)
                    print(f"    ✔ Row {idx}: downloaded → {final_path}")
                except Exception as e:
                    print(f"    ✖ Row {idx}: failed to download image: {e}")
            else:
                print(f"    ✖ Row {idx}: could not extract Drive ID")
        else:
            print(f"    (Row {idx}) No image link found.")

print("[+] Done!")


[+] Downloading CSV...
[+] Saved CSV → complaints.csv
[+] Processing rows...
    ✔ Row 1: downloaded → downloaded_images\sanitation_trash_issue_2026-07-08.jpg
[+] Done!


In [4]:
import csv
import os
import re
import requests
from datetime import datetime

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

SHEET_URL = "https://docs.google.com/spreadsheets/d/1lHUX4W8M2yKQSymul4Xu55jUri9fzgK2h7lHmQdPQoQ/export?format=csv"

BASE_DIR = "data"
MASTER_CSV = os.path.join(BASE_DIR, "master.csv")
REGISTRY_CSV = os.path.join(BASE_DIR, "image_registry.csv")

os.makedirs(BASE_DIR, exist_ok=True)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_slug(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text or "none"

def extract_drive_id(url):
    if "id=" in url:
        return url.split("id=")[1]
    m = re.search(r"/d/([^/]+)", url)
    if m:
        return m.group(1)
    return None

def download_drive_file(file_id, dest_path):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return dest_path

def next_available_filename(base_path):
    if not os.path.exists(base_path):
        return base_path
    root, ext = os.path.splitext(base_path)
    counter = 1
    while True:
        candidate = f"{root}_{counter}{ext}"
        if not os.path.exists(candidate):
            return candidate
        counter += 1

# ------------------------------------------------------------
# LOAD OR CREATE IMAGE REGISTRY
# ------------------------------------------------------------

registry = {}

if os.path.exists(REGISTRY_CSV):
    with open(REGISTRY_CSV, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            registry[row["file_id"]] = row["saved_path"]
else:
    with open(REGISTRY_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file_id", "saved_path"])
        writer.writeheader()

# ------------------------------------------------------------
# STEP 1 — DOWNLOAD MASTER CSV
# ------------------------------------------------------------

print("[+] Downloading master CSV...")
csv_data = requests.get(SHEET_URL)
csv_data.raise_for_status()

with open(MASTER_CSV, "wb") as f:
    f.write(csv_data.content)

print(f"[+] Saved master CSV → {MASTER_CSV}")

# ------------------------------------------------------------
# STEP 2 — PROCESS ROWS
# ------------------------------------------------------------

print("[+] Processing rows...")

with open(MASTER_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for idx, row in enumerate(reader, start=1):

        # Extract fields
        address = safe_slug(row.get("Address of Rental Property", "unknown"))
        unit = safe_slug(row.get("Unit/Apartment Number", "none"))
        nature = safe_slug(row.get("Nature of Complaint", "unknown"))
        date_incident = row.get("Date of Incident", "").strip()

        # Normalize date
        try:
            parsed_date = datetime.strptime(date_incident, "%m/%d/%Y")
            date_slug = parsed_date.strftime("%Y-%m-%d")
        except:
            date_slug = "unknown_date"

        # Folder structure
        addr_dir = os.path.join(BASE_DIR, address)
        unit_dir = os.path.join(addr_dir, unit)
        os.makedirs(unit_dir, exist_ok=True)

        # Per-unit CSV
        unit_csv = os.path.join(unit_dir, "complaints.csv")
        new_file = not os.path.exists(unit_csv)

        with open(unit_csv, "a", encoding="utf-8", newline="") as uf:
            writer = csv.DictWriter(uf, fieldnames=reader.fieldnames)
            if new_file:
                writer.writeheader()
            writer.writerow(row)

        # Handle image
        img_url = row.get("Upload Images or Documents", "").strip()
        if not img_url:
            print(f"    (Row {idx}) No image link.")
            continue

        file_id = extract_drive_id(img_url)
        if not file_id:
            print(f"    ✖ Row {idx}: Could not extract Drive ID")
            continue

        # Check registry
        if file_id in registry:
            print(f"    ✔ Row {idx}: Image already downloaded → {registry[file_id]}")
            continue

        # Build filename
        base_filename = f"{nature}_{date_slug}.jpg"
        base_path = os.path.join(unit_dir, base_filename)
        final_path = next_available_filename(base_path)

        # Download
        try:
            download_drive_file(file_id, final_path)
            print(f"    ✔ Row {idx}: Downloaded → {final_path}")

            # Update registry
            registry[file_id] = final_path
            with open(REGISTRY_CSV, "a", encoding="utf-8", newline="") as rf:
                writer = csv.DictWriter(rf, fieldnames=["file_id", "saved_path"])
                writer.writerow({"file_id": file_id, "saved_path": final_path})

        except Exception as e:
            print(f"    ✖ Row {idx}: Failed to download image: {e}")

print("[+] Done!")


[+] Downloading master CSV...
[+] Saved master CSV → data\master.csv
[+] Processing rows...
    ✔ Row 1: Image already downloaded → data\9240_garland_dr_savannah_ga_31406\none\sanitation_trash_issue_2026-07-08.jpg
[+] Done!
